In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder.master("local[*]").appName("PySpark Assignment").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# --- Employees ---
employees_data = [
    (1,  "Alice",   "Engineering", "India",  95000, "2019-03-15", True),
    (2,  "Bob",     "Marketing",   "USA",    72000, "2020-07-01", True),
    (3,  "Carol",   "Engineering", "USA",    110000,"2018-11-20", True),
    (4,  "David",   "HR",          "India",  60000, "2021-01-10", False),
    (5,  "Eva",     "Engineering", "UK",     98000, "2017-06-05", True),
    (6,  "Frank",   "Marketing",   "India",  68000, "2022-03-22", True),
    (7,  "Grace",   "HR",          "USA",    63000, "2020-09-14", True),
    (8,  "Hank",    "Engineering", "UK",     120000,"2016-12-01", False),
    (9,  "Ivy",     "Marketing",   "UK",     75000, "2021-05-30", True),
    (10, "Jake",    "HR",          "USA",    58000, "2023-02-18", True),
]
employees_schema = StructType([
    StructField("emp_id",     IntegerType(), False),
    StructField("name",       StringType(),  False),
    StructField("department", StringType(),  False),
    StructField("country",    StringType(),  False),
    StructField("salary",     IntegerType(), False),
    StructField("join_date",  StringType(),  False),
    StructField("is_active",  BooleanType(), False),
])
employees = spark.createDataFrame(employees_data, employees_schema)

# --- Orders ---
orders_data = [
    (101, 1, "2024-01-05", 500.0,  "Electronics"),
    (102, 3, "2024-01-18", 1200.0, "Electronics"),
    (103, 2, "2024-02-10", 300.0,  "Clothing"),
    (104, 1, "2024-02-20", 850.0,  "Electronics"),
    (105, 5, "2024-03-01", 2000.0, "Furniture"),
    (106, 3, "2024-03-15", 450.0,  "Clothing"),
    (107, 7, "2024-03-22", 150.0,  "Clothing"),
    (108, 2, "2024-04-10", 700.0,  "Electronics"),
    (109, 9, "2024-04-25", 950.0,  "Furniture"),
    (110, 1, "2024-05-02", 1100.0, "Electronics"),
    (111, 4, "2024-05-15", 200.0,  "Clothing"),
    (112, 6, "2024-06-01", 600.0,  "Electronics"),
]
orders_schema = StructType([
    StructField("order_id",   IntegerType(), False),
    StructField("emp_id",     IntegerType(), False),
    StructField("order_date", StringType(),  False),
    StructField("amount",     DoubleType(),  False),
    StructField("category",   StringType(),  False),
])
orders = spark.createDataFrame(orders_data, orders_schema)

## Part 1 — DataFrame Basics & Schema

### 1.1 Print the schema of employees and orders using printSchema().

In [23]:
orders.printSchema()

root
 |-- order_id: integer (nullable = false)
 |-- emp_id: integer (nullable = false)
 |-- order_date: string (nullable = false)
 |-- amount: double (nullable = false)
 |-- category: string (nullable = false)



### 1.2 Cast join_date (currently a string) in employees to DateType. Store the result back as employees.

In [24]:
employees = employees.withColumn("join_date", col('join_date').cast(DateType()))
employees.printSchema()

root
 |-- emp_id: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- department: string (nullable = false)
 |-- country: string (nullable = false)
 |-- salary: integer (nullable = false)
 |-- join_date: date (nullable = true)
 |-- is_active: boolean (nullable = false)



### 1.3 Cast order_date in orders to DateType. Store back as orders.

In [25]:
orders = orders.withColumn("order_date", col('order_date').cast(DateType()))
orders.printSchema()

root
 |-- order_id: integer (nullable = false)
 |-- emp_id: integer (nullable = false)
 |-- order_date: date (nullable = true)
 |-- amount: double (nullable = false)
 |-- category: string (nullable = false)



### 1.4 Add a column years_of_service to employees that computes the number of full years between join_date and today (current_date()).

In [26]:
employees = employees.withColumn('years_of_service', (year(current_date()) - year(col('join_date'))))
employees.show()

+------+-----+-----------+-------+------+----------+---------+----------------+
|emp_id| name| department|country|salary| join_date|is_active|years_of_service|
+------+-----+-----------+-------+------+----------+---------+----------------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|               7|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|               6|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|               8|
|     4|David|         HR|  India| 60000|2021-01-10|    false|               5|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|               9|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|               4|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|               6|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|              10|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|               5|
|    10| Jake|         HR|    USA| 58000

### 1.5 Add a boolean column is_senior that is True when years_of_service >= 5.

In [27]:
employees = employees.withColumn('is_senior', when(col('years_of_service') >= 5, True).otherwise(False))
employees.show()

+------+-----+-----------+-------+------+----------+---------+----------------+---------+
|emp_id| name| department|country|salary| join_date|is_active|years_of_service|is_senior|
+------+-----+-----------+-------+------+----------+---------+----------------+---------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|               7|     true|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|               6|     true|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|               8|     true|
|     4|David|         HR|  India| 60000|2021-01-10|    false|               5|     true|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|               9|     true|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|               4|    false|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|               6|     true|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|              10|     true|
|     9|  

## Part 2 — Filtering & Column Operations

### 2.1 Select all active employees (is_active = True) in the Engineering department. Show only name, salary, and country.

In [28]:
employees \
    .filter((col('is_active') == True) & (col('department') == 'Engineering')) \
    .select('name', 'salary', 'country') \
    .show()

+-----+------+-------+
| name|salary|country|
+-----+------+-------+
|Alice| 95000|  India|
|Carol|110000|    USA|
|  Eva| 98000|     UK|
+-----+------+-------+



### 2.2 Find all employees with a salary between 70,000 and 100,000 (inclusive). Sort by salary descending.

In [29]:
employees \
    .filter(col('salary').between(70000, 100000)) \
    .sort(col('salary').desc()) \
    .show()

+------+-----+-----------+-------+------+----------+---------+----------------+---------+
|emp_id| name| department|country|salary| join_date|is_active|years_of_service|is_senior|
+------+-----+-----------+-------+------+----------+---------+----------------+---------+
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|               9|     true|
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|               7|     true|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|               5|     true|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|               6|     true|
+------+-----+-----------+-------+------+----------+---------+----------------+---------+



### 2.3 Add a column salary_band:
### "Low" if salary < 65,000
### "Mid" if 65,000 ≤ salary < 100,000
### "High" if salary ≥ 100,000
### Use when / otherwise.

In [30]:
employees = employees \
    .withColumn('salary_band', \
        when(col('salary') < 65000, "Low") \
        .when(col('salary').between(65000, 100000 - 1), "Mid") \
        .otherwise("High")
    )

### 2.4 Rename name to employee_name and drop the is_active column. Show the resulting schema.

In [180]:
employees = employees \
    .withColumnRenamed("name","employee_name") \
    .drop("is_active")
employees.show()

+------+-------------+-----------+-------+------+----------+
|emp_id|employee_name| department|country|salary| join_date|
+------+-------------+-----------+-------+------+----------+
|     1|        Alice|Engineering|  India| 95000|2019-03-15|
|     2|          Bob|  Marketing|    USA| 72000|2020-07-01|
|     3|        Carol|Engineering|    USA|110000|2018-11-20|
|     4|        David|         HR|  India| 60000|2021-01-10|
|     5|          Eva|Engineering|     UK| 98000|2017-06-05|
|     6|        Frank|  Marketing|  India| 68000|2022-03-22|
|     7|        Grace|         HR|    USA| 63000|2020-09-14|
|     8|         Hank|Engineering|     UK|120000|2016-12-01|
|     9|          Ivy|  Marketing|     UK| 75000|2021-05-30|
|    10|         Jake|         HR|    USA| 58000|2023-02-18|
+------+-------------+-----------+-------+------+----------+



## Part 3 — Aggregations & GroupBy

### 3.1 Calculate the average salary per department. Round to 2 decimal places.

In [32]:
employees \
    .groupBy('department') \
    .agg(round(avg('salary'), 2).alias('avg_salary')) \
    .show()

+-----------+----------+
| department|avg_salary|
+-----------+----------+
|Engineering|  105750.0|
|  Marketing|  71666.67|
|         HR|  60333.33|
+-----------+----------+



### 3.2 For each department, find:

#### Total headcount
#### Number of active employees
#### Minimum and maximum salary

In [35]:
employees \
    .groupBy('department') \
    .agg(
        count('emp_id').alias('total_headcount'),
        count(when(col('is_active') == True, col('emp_id'))).alias('active_emps'),
        min('salary').alias('min_salary'),
        max('salary').alias('max_salary')
    ).show()

+-----------+---------------+-----------+----------+----------+
| department|total_headcount|active_emps|min_salary|max_salary|
+-----------+---------------+-----------+----------+----------+
|Engineering|              4|          3|     95000|    120000|
|  Marketing|              3|          3|     68000|     75000|
|         HR|              3|          2|     58000|     63000|
+-----------+---------------+-----------+----------+----------+



### 3.3 Find departments where the average salary is above 80,000.

In [40]:
employees \
    .groupBy('department') \
    .agg(avg('salary').alias('avg_salary')) \
    .filter(col('avg_salary') > 80000) \
    .show()

+-----------+----------+
| department|avg_salary|
+-----------+----------+
|Engineering|  105750.0|
+-----------+----------+



### 3.4 From orders, compute total revenue and average order value per category. Sort by total revenue descending.

In [42]:
orders \
    .groupBy('category') \
    .agg( \
        sum('amount').alias('total_revenue'), \
        avg('amount').alias('avg_order_value') \
    ) \
    .sort(col('total_revenue').desc()) \
    .show()

+-----------+-------------+---------------+
|   category|total_revenue|avg_order_value|
+-----------+-------------+---------------+
|Electronics|       4950.0|          825.0|
|  Furniture|       2950.0|         1475.0|
|   Clothing|       1100.0|          275.0|
+-----------+-------------+---------------+



## Part 4 — Joins

### 4.1 Perform an inner join of orders with employees on emp_id. Select: order_id, employee_name (renamed from name), department, amount, category, order_date.

In [45]:
df = orders \
        .join(employees, orders['emp_id'] == employees['emp_id']) \
        .select('order_id', 'employee_name', 'department', 'amount', 'category', 'order_date')
df.show()

[Stage 67:>                                                         (0 + 8) / 8]

+--------+-------------+-----------+------+-----------+----------+
|order_id|employee_name| department|amount|   category|order_date|
+--------+-------------+-----------+------+-----------+----------+
|     101|        Alice|Engineering| 500.0|Electronics|2024-01-05|
|     104|        Alice|Engineering| 850.0|Electronics|2024-02-20|
|     110|        Alice|Engineering|1100.0|Electronics|2024-05-02|
|     103|          Bob|  Marketing| 300.0|   Clothing|2024-02-10|
|     108|          Bob|  Marketing| 700.0|Electronics|2024-04-10|
|     102|        Carol|Engineering|1200.0|Electronics|2024-01-18|
|     106|        Carol|Engineering| 450.0|   Clothing|2024-03-15|
|     111|        David|         HR| 200.0|   Clothing|2024-05-15|
|     105|          Eva|Engineering|2000.0|  Furniture|2024-03-01|
|     112|        Frank|  Marketing| 600.0|Electronics|2024-06-01|
|     107|        Grace|         HR| 150.0|   Clothing|2024-03-22|
|     109|          Ivy|  Marketing| 950.0|  Furniture|2024-04

### 4.2 Find employees who have placed no orders (use a left anti join or left join + filter on null).

In [47]:
df = employees \
        .join(orders, orders['emp_id'] == employees['emp_id'], 'left_anti')
df.show()

+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+
|emp_id|employee_name| department|country|salary| join_date|is_active|years_of_service|is_senior|salary_band|
+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+
|     8|         Hank|Engineering|     UK|120000|2016-12-01|    false|              10|     true|       High|
|    10|         Jake|         HR|    USA| 58000|2023-02-18|     true|               3|    false|        Low|
+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+



### 4.3 Find the top spender (employee with the highest total order amount). Show their name, department, and total amount.

In [69]:
df = employees \
        .join(orders, orders['emp_id'] == employees['emp_id']) \
        .groupBy(orders['emp_id'], 'employee_name', 'department') \
        .agg(sum('amount').alias('total_amount')) \
        .select('employee_name', 'department', 'total_amount') \
        .withColumn('rnk', dense_rank().over(Window.orderBy(col('total_amount').desc())))
df.filter(col('rnk') == 1).show()

[Stage 128:>                                                        (0 + 8) / 8]

+-------------+-----------+------------+---+
|employee_name| department|total_amount|rnk|
+-------------+-----------+------------+---+
|        Alice|Engineering|      2450.0|  1|
+-------------+-----------+------------+---+



## Part 5 — Window Functions

### 5.1 Rank employees within each department by salary (highest = rank 1). Use rank(). Show name, department, salary, and rank.

In [70]:
df = employees \
        .select('employee_name', 'department', 'salary') \
        .withColumn('rnk', rank().over(Window.orderBy(col('salary').desc())))
df.filter(col('rnk') == 1).show()

+-------------+-----------+------+---+
|employee_name| department|salary|rnk|
+-------------+-----------+------+---+
|         Hank|Engineering|120000|  1|
+-------------+-----------+------+---+



### 5.2 For each employee, compute the cumulative salary sum within their department, ordered by join_date ascending. Use sum() as a window function.

In [81]:
employees \
        .select('employee_name', 'department', 'salary', 'join_date') \
        .withColumn('cum_salary', sum('salary').over(Window.partitionBy('department').orderBy(col('join_date')))) \
        .sort('department', 'join_date') \
        .show()

+-------------+-----------+------+----------+----------+
|employee_name| department|salary| join_date|cum_salary|
+-------------+-----------+------+----------+----------+
|         Hank|Engineering|120000|2016-12-01|    120000|
|          Eva|Engineering| 98000|2017-06-05|    218000|
|        Carol|Engineering|110000|2018-11-20|    328000|
|        Alice|Engineering| 95000|2019-03-15|    423000|
|        Grace|         HR| 63000|2020-09-14|     63000|
|        David|         HR| 60000|2021-01-10|    123000|
|         Jake|         HR| 58000|2023-02-18|    181000|
|          Bob|  Marketing| 72000|2020-07-01|     72000|
|          Ivy|  Marketing| 75000|2021-05-30|    147000|
|        Frank|  Marketing| 68000|2022-03-22|    215000|
+-------------+-----------+------+----------+----------+



### 5.3 For each order, add a column running_total — the running total of amount per emp_id ordered by order_date.

In [86]:
employees \
    .join(orders, 'emp_id') \
    .select('employee_name', 'order_date', 'amount') \
    .withColumn('running_total', sum('amount').over(Window.partitionBy(orders['emp_id']).orderBy('order_date'))) \
    .sort('employee_name', 'order_date') \
    .show()

[Stage 173:>                                                        (0 + 8) / 8]

+-------------+----------+------+-------------+
|employee_name|order_date|amount|running_total|
+-------------+----------+------+-------------+
|        Alice|2024-01-05| 500.0|        500.0|
|        Alice|2024-02-20| 850.0|       1350.0|
|        Alice|2024-05-02|1100.0|       2450.0|
|          Bob|2024-02-10| 300.0|        300.0|
|          Bob|2024-04-10| 700.0|       1000.0|
|        Carol|2024-01-18|1200.0|       1200.0|
|        Carol|2024-03-15| 450.0|       1650.0|
|        David|2024-05-15| 200.0|        200.0|
|          Eva|2024-03-01|2000.0|       2000.0|
|        Frank|2024-06-01| 600.0|        600.0|
|        Grace|2024-03-22| 150.0|        150.0|
|          Ivy|2024-04-25| 950.0|        950.0|
+-------------+----------+------+-------------+



### 5.4 Add a column prev_order_amount to orders showing the previous order amount for the same employee (use lag()). Fill nulls with 0.0.

In [93]:
orders \
    .withColumn('prev_order_amount', lag('amount').over(Window.partitionBy('emp_id').orderBy('order_date'))) \
    .fillna(0.0, subset=['prev_order_amount']) \
    .show()

+--------+------+----------+------+-----------+-----------------+
|order_id|emp_id|order_date|amount|   category|prev_order_amount|
+--------+------+----------+------+-----------+-----------------+
|     101|     1|2024-01-05| 500.0|Electronics|              0.0|
|     104|     1|2024-02-20| 850.0|Electronics|            500.0|
|     110|     1|2024-05-02|1100.0|Electronics|            850.0|
|     103|     2|2024-02-10| 300.0|   Clothing|              0.0|
|     108|     2|2024-04-10| 700.0|Electronics|            300.0|
|     102|     3|2024-01-18|1200.0|Electronics|              0.0|
|     106|     3|2024-03-15| 450.0|   Clothing|           1200.0|
|     111|     4|2024-05-15| 200.0|   Clothing|              0.0|
|     105|     5|2024-03-01|2000.0|  Furniture|              0.0|
|     112|     6|2024-06-01| 600.0|Electronics|              0.0|
|     107|     7|2024-03-22| 150.0|   Clothing|              0.0|
|     109|     9|2024-04-25| 950.0|  Furniture|              0.0|
+--------+

## Part 6 — String & Date Functions

### 6.1 Add a column name_upper (uppercase of name) and dept_short (first 3 characters of department, uppercased).

In [101]:
employees = employees \
                .withColumn('name_upper', upper(col('employee_name'))) \
                .withColumn('dept_short', upper(substring(col('department'),1,3)))
employees.show()

+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+
|emp_id|employee_name| department|country|salary| join_date|is_active|years_of_service|is_senior|salary_band|name_upper|dept_short|
+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+
|     1|        Alice|Engineering|  India| 95000|2019-03-15|     true|               7|     true|        Mid|     ALICE|       ENG|
|     2|          Bob|  Marketing|    USA| 72000|2020-07-01|     true|               6|     true|        Mid|       BOB|       MAR|
|     3|        Carol|Engineering|    USA|110000|2018-11-20|     true|               8|     true|       High|     CAROL|       ENG|
|     4|        David|         HR|  India| 60000|2021-01-10|    false|               5|     true|        Low|     DAVID|        HR|
|     5|          Eva|Engineering|     UK| 98000|2017-06-05|     true|      

### 6.2 Extract join_year, join_month from join_date as separate integer columns.

In [105]:
employees \
    .withColumn('join_year', year(col('join_date'))) \
    .withColumn('join_month', month(col('join_date'))) \
    .select('employee_name', 'join_year', 'join_month') \
    .show()

+-------------+---------+----------+
|employee_name|join_year|join_month|
+-------------+---------+----------+
|        Alice|     2019|         3|
|          Bob|     2020|         7|
|        Carol|     2018|        11|
|        David|     2021|         1|
|          Eva|     2017|         6|
|        Frank|     2022|         3|
|        Grace|     2020|         9|
|         Hank|     2016|        12|
|          Ivy|     2021|         5|
|         Jake|     2023|         2|
+-------------+---------+----------+



### 6.3 From orders, extract the order_month and find the month with the highest total revenue.

In [109]:
orders \
    .withColumn('order_month', month(col('order_date'))) \
    .groupBy('order_month') \
    .agg(sum('amount').alias('total_revenue')) \
    .withColumn('rnk', dense_rank().over(Window.orderBy(col('total_revenue').desc()))) \
    .filter(col('rnk') == 1) \
    .show()

+-----------+-------------+---+
|order_month|total_revenue|rnk|
+-----------+-------------+---+
|          3|       2600.0|  1|
+-----------+-------------+---+



### 6.4 Add a column name_length (character count of name) and filter to employees whose name is longer than 4 characters.

In [114]:
employees \
    .withColumn('name_length', length(col('employee_name'))) \
    .filter(col('name_length') > 4) \
    .select('employee_name', 'name_length') \
    .show()

+-------------+-----------+
|employee_name|name_length|
+-------------+-----------+
|        Alice|          5|
|        Carol|          5|
|        David|          5|
|        Frank|          5|
|        Grace|          5|
+-------------+-----------+



## Part 7 — SQL Queries

In [115]:
employees.createOrReplaceTempView("employees")
orders.createOrReplaceTempView("orders")

### 7.1 Write a SQL query to get the top 3 highest-paid employees per department.

In [120]:
spark.sql(
    """
    SELECT *
    FROM (
        SELECT
            emp_id,
            employee_name,
            department,
            salary,
            DENSE_RANK() OVER(PARTITION BY department ORDER BY salary DESC) AS rnk
        FROM employees
    ) AS tbl
    WHERE tbl.rnk <= 3
    """
).show()

+------+-------------+-----------+------+---+
|emp_id|employee_name| department|salary|rnk|
+------+-------------+-----------+------+---+
|     8|         Hank|Engineering|120000|  1|
|     3|        Carol|Engineering|110000|  2|
|     5|          Eva|Engineering| 98000|  3|
|     7|        Grace|         HR| 63000|  1|
|     4|        David|         HR| 60000|  2|
|    10|         Jake|         HR| 58000|  3|
|     9|          Ivy|  Marketing| 75000|  1|
|     2|          Bob|  Marketing| 72000|  2|
|     6|        Frank|  Marketing| 68000|  3|
+------+-------------+-----------+------+---+



### 7.2 Write a SQL query to find all employees from India or UK who joined before 2021.

In [207]:
spark.sql(
    """
    SELECT
        employee_name,
        country,
        EXTRACT(YEAR FROM join_date) AS join_year
    FROM employees
    WHERE
        country IN ("India", "UK") AND
        EXTRACT(YEAR FROM join_date) < 2021
    """
).show()

+-------------+-------+---------+
|employee_name|country|join_year|
+-------------+-------+---------+
|        Alice|  India|     2019|
|          Eva|     UK|     2017|
|         Hank|     UK|     2016|
+-------------+-------+---------+



### 7.3 Write a SQL query that joins employees and orders, groups by department and category, and returns total revenue per combination. Sort by department, then total_revenue descending.

In [126]:
spark.sql(
    """
    SELECT
        e.department,
        o.category,
        SUM(o.amount) AS total_revenue
    FROM employees e
    JOIN orders o USING(emp_id)
    GROUP BY e.department, o.category
    ORDER BY e.department, total_revenue DESC
    """
).show()

[Stage 235:=================================================>       (7 + 1) / 8]

+-----------+-----------+-------------+
| department|   category|total_revenue|
+-----------+-----------+-------------+
|Engineering|Electronics|       3650.0|
|Engineering|  Furniture|       2000.0|
|Engineering|   Clothing|        450.0|
|         HR|   Clothing|        350.0|
|  Marketing|Electronics|       1300.0|
|  Marketing|  Furniture|        950.0|
|  Marketing|   Clothing|        300.0|
+-----------+-----------+-------------+



## Part 8 — Handling Nulls & Duplicates

In [155]:
new_schema = StructType([
    StructField(emp.name, emp.dataType, True) if emp.name in ['department', 'country', 'salary' , 'join_date'] else emp for emp in employees_schema
])
employees = spark.createDataFrame(employees_data, new_schema)

In [156]:
# Introduce some nulls for this section
dirty_data = [
    (11, "Leo",   None,          "USA",  None,  "2020-01-01", True),
    (12, "Mia",   "Engineering", None,   88000, None,         True),
    (1,  "Alice", "Engineering", "India",95000, "2019-03-15", True),  # duplicate
]
dirty_df = spark.createDataFrame(dirty_data, new_schema)
employees_dirty = employees.union(dirty_df)

### 8.1 Count the total nulls in each column of employees_dirty.

In [167]:
employees_dirty.select([
    count(when(col(cn).isNull(), 1)).alias(cn)
    for cn in employees_dirty.columns
]).show()

+------+----+----------+-------+------+---------+---------+
|emp_id|name|department|country|salary|join_date|is_active|
+------+----+----------+-------+------+---------+---------+
|     0|   0|         1|      1|     1|        1|        0|
+------+----+----------+-------+------+---------+---------+



### 8.2 Fill null department with "Unknown" and null salary with the median salary of the existing employees (compute it first, then use fillna).

In [174]:
med_salary = employees_dirty.agg(median('salary')).collect()[0][0]
employees_dirty = employees_dirty \
                        .fillna({ 
                            'department' : "Unknown",
                            'salary' : med_salary
                        })
employees_dirty.show()

+------+-----+-----------+-------+------+----------+---------+
|emp_id| name| department|country|salary| join_date|is_active|
+------+-----+-----------+-------+------+----------+---------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|
|     4|David|         HR|  India| 60000|2021-01-10|    false|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|
|    10| Jake|         HR|    USA| 58000|2023-02-18|     true|
|    11|  Leo|    Unknown|    USA| 81500|2020-01-01|     true|
|    12|  Mia|Engineering|   NULL| 88000|      NULL|     true|
|     1|Alice|Engineering|  India| 95000|2019-03-15|   

### 8.3 Drop rows where country is null.

In [177]:
employees_dirty = employees_dirty.na.drop(subset=['country'])
employees_dirty.show()

+------+-----+-----------+-------+------+----------+---------+
|emp_id| name| department|country|salary| join_date|is_active|
+------+-----+-----------+-------+------+----------+---------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|
|     4|David|         HR|  India| 60000|2021-01-10|    false|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|
|    10| Jake|         HR|    USA| 58000|2023-02-18|     true|
|    11|  Leo|    Unknown|    USA| 81500|2020-01-01|     true|
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|
+------+-----+-----------+-------+------+----------+---

### 8.4 Remove duplicate rows based on emp_id, keeping the first occurrence.

In [178]:
employees_cleaned = employees_dirty.drop_duplicates(subset=['emp_id'])
employees_cleaned.show()

[Stage 285:============================>                           (8 + 8) / 16]

+------+-----+-----------+-------+------+----------+---------+
|emp_id| name| department|country|salary| join_date|is_active|
+------+-----+-----------+-------+------+----------+---------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|
|     4|David|         HR|  India| 60000|2021-01-10|    false|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|
|    10| Jake|         HR|    USA| 58000|2023-02-18|     true|
|    11|  Leo|    Unknown|    USA| 81500|2020-01-01|     true|
+------+-----+-----------+-------+------+----------+---------+



## Part 9 — Persisting & Partitioning (local filesystem)

### 9.1 Write the joined DataFrame from Task 4.1 to /tmp/pyspark_assignment/orders_enriched/ as Parquet, partitioned by department.

In [184]:
df = orders \
        .join(employees, orders['emp_id'] == employees['emp_id']) \
        .select('order_id', 'employee_name', 'department', 'amount', 'category', 'order_date')

In [186]:
df.write \
    .mode('overwrite') \
    .partitionBy('department') \
    .parquet('/tmp/pyspark_assignment/orders_enriched')

### 9.2 Read the Parquet files back and verify the row count matches the original.

In [190]:
orders_par = spark.read.parquet('/tmp/pyspark_assignment/orders_enriched')
orders_par.show()

+--------+-------------+------+-----------+----------+-----------+
|order_id|employee_name|amount|   category|order_date| department|
+--------+-------------+------+-----------+----------+-----------+
|     101|        Alice| 500.0|Electronics|2024-01-05|Engineering|
|     104|        Alice| 850.0|Electronics|2024-02-20|Engineering|
|     110|        Alice|1100.0|Electronics|2024-05-02|Engineering|
|     102|        Carol|1200.0|Electronics|2024-01-18|Engineering|
|     106|        Carol| 450.0|   Clothing|2024-03-15|Engineering|
|     105|          Eva|2000.0|  Furniture|2024-03-01|Engineering|
|     103|          Bob| 300.0|   Clothing|2024-02-10|  Marketing|
|     108|          Bob| 700.0|Electronics|2024-04-10|  Marketing|
|     112|        Frank| 600.0|Electronics|2024-06-01|  Marketing|
|     109|          Ivy| 950.0|  Furniture|2024-04-25|  Marketing|
|     111|        David| 200.0|   Clothing|2024-05-15|         HR|
|     107|        Grace| 150.0|   Clothing|2024-03-22|        

### 9.3 Write the employees DataFrame as a single CSV file (use coalesce(1)) with a header to /tmp/pyspark_assignment/employees.csv.

In [191]:
employees_cleaned.coalesce(1).write \
    .option("header", "true") \
    .mode('overwrite') \
    .csv('/tmp/pyspark_assignment/employees')

### 9.4 Read the CSV back with inferSchema=True and print its schema.

In [194]:
employees_csv = spark.read.csv('/tmp/pyspark_assignment/employees', inferSchema=True)
employees_csv.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)



## Part 10 — UDFs

### 10.1 Write a Python UDF categorize_salary(salary) that returns:

#### "Entry" if salary < 65,000
#### "Mid" if 65,000 ≤ salary < 100,000
#### "Senior" if salary ≥ 100,000
#### Register it and apply it to the employees DataFrame.

In [197]:
def categorize_salary(salary):
    if salary < 65000:
        return "Entry"
    elif salary >= 65000 and salary < 100000:
        return "Mid"
    else:
        return "Senior"

categorize_salary_udf = udf(categorize_salary, StringType())

employees_cleaned \
    .withColumn('category', categorize_salary_udf(col('salary'))) \
    .show()

[Stage 329:===============================>                        (9 + 7) / 16]

+------+-----+-----------+-------+------+----------+---------+--------+
|emp_id| name| department|country|salary| join_date|is_active|category|
+------+-----+-----------+-------+------+----------+---------+--------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|     Mid|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|     Mid|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|  Senior|
|     4|David|         HR|  India| 60000|2021-01-10|    false|   Entry|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|     Mid|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|     Mid|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|   Entry|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|  Senior|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|     Mid|
|    10| Jake|         HR|    USA| 58000|2023-02-18|     true|   Entry|
|    11|  Leo|    Unknown|    USA| 81500|2020-01-01|     true|  

### 10.2 Write a UDF mask_name(name) that returns the first character followed by asterisks for the rest (e.g. "Alice" → "A****"). Apply it to employees.

In [206]:
def mask_name(name):
    return name[0] + "*" * (len(name) - 1)

mask_name_udf = udf(mask_name, StringType())

employees_cleaned \
    .withColumn('name', mask_name_udf(col('name'))) \
    .show()

+------+-----+-----------+-------+------+----------+---------+
|emp_id| name| department|country|salary| join_date|is_active|
+------+-----+-----------+-------+------+----------+---------+
|     1|A****|Engineering|  India| 95000|2019-03-15|     true|
|     2|  B**|  Marketing|    USA| 72000|2020-07-01|     true|
|     3|C****|Engineering|    USA|110000|2018-11-20|     true|
|     4|D****|         HR|  India| 60000|2021-01-10|    false|
|     5|  E**|Engineering|     UK| 98000|2017-06-05|     true|
|     6|F****|  Marketing|  India| 68000|2022-03-22|     true|
|     7|G****|         HR|    USA| 63000|2020-09-14|     true|
|     8| H***|Engineering|     UK|120000|2016-12-01|    false|
|     9|  I**|  Marketing|     UK| 75000|2021-05-30|     true|
|    10| J***|         HR|    USA| 58000|2023-02-18|     true|
|    11|  L**|    Unknown|    USA| 81500|2020-01-01|     true|
+------+-----+-----------+-------+------+----------+---------+

